In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SCNE Regression Models") \
    .master("local[*]") \
    .getOrCreate()

DATA_PATH = r"D:\Big Data Programming Project\Final Assignment\data\processed\spark\scne_model_features"

model_df = spark.read.parquet(DATA_PATH)

print("Rows:", model_df.count())
print("Columns:", len(model_df.columns))

Rows: 308885
Columns: 16


In [3]:
train_df = model_df.filter(
    model_df.service_date.isin("2025-12-26", "2025-12-27")
)

test_df = model_df.filter(
    model_df.service_date == "2025-12-28"
)

print("Training rows:", train_df.count())
print("Testing rows:", test_df.count())

Training rows: 202040
Testing rows: 106845


In [4]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml import Pipeline

route_indexer = StringIndexer(
    inputCol="published_line_name",
    outputCol="route_index",
    handleInvalid="keep"
)

stop_indexer = StringIndexer(
    inputCol="stop_id",
    outputCol="stop_index",
    handleInvalid="keep"
)

feature_columns = [
    "route_index",
    "stop_index",
    "direction_id",
    "stop_sequence",
    "hour",
    "minute",
    "day_of_week",
    "is_weekend",
    "is_public_holiday",
    "journey_progress",
    "previous_stop_delay",
    "rolling_previous_delay",
    "has_previous_delay"
]

assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

prep_pipeline = Pipeline(
    stages=[
        route_indexer,
        stop_indexer,
        assembler
    ]
)

prep_model = prep_pipeline.fit(train_df)

train_ready = prep_model.transform(train_df)
test_ready = prep_model.transform(test_df)

print("Training prepared:", train_ready.count())
print("Testing prepared:", test_ready.count())
print("Number of features:", len(feature_columns))

Training prepared: 202040
Testing prepared: 106845
Number of features: 13


In [5]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

lr = LinearRegression(
    featuresCol="features",
    labelCol="delay_seconds"
)

lr_model = lr.fit(train_ready)

lr_predictions = lr_model.transform(test_ready)

mae_evaluator = RegressionEvaluator(
    labelCol="delay_seconds",
    predictionCol="prediction",
    metricName="mae"
)

rmse_evaluator = RegressionEvaluator(
    labelCol="delay_seconds",
    predictionCol="prediction",
    metricName="rmse"
)

r2_evaluator = RegressionEvaluator(
    labelCol="delay_seconds",
    predictionCol="prediction",
    metricName="r2"
)

lr_mae = mae_evaluator.evaluate(lr_predictions)
lr_rmse = rmse_evaluator.evaluate(lr_predictions)
lr_r2 = r2_evaluator.evaluate(lr_predictions)

print("Linear Regression")
print("MAE:", round(lr_mae, 2))
print("RMSE:", round(lr_rmse, 2))
print("R²:", round(lr_r2, 4))

Linear Regression
MAE: 93.34
RMSE: 149.36
R²: 0.711


In [6]:
print(
    "Training routes:",
    train_df.select("published_line_name").distinct().count()
)

print(
    "Training stops:",
    train_df.select("stop_id").distinct().count()
)

Training routes: 62
Training stops: 3989


In [7]:
tree_feature_columns = [
    "route_index",
    "direction_id",
    "stop_sequence",
    "hour",
    "minute",
    "day_of_week",
    "is_weekend",
    "is_public_holiday",
    "journey_progress",
    "previous_stop_delay",
    "rolling_previous_delay",
    "has_previous_delay"
]

tree_assembler = VectorAssembler(
    inputCols=tree_feature_columns,
    outputCol="tree_features"
)

train_tree = tree_assembler.transform(train_ready)
test_tree = tree_assembler.transform(test_ready)

print("Tree features:", len(tree_feature_columns))

Tree features: 12


In [8]:
from pyspark.ml.regression import DecisionTreeRegressor

dt = DecisionTreeRegressor(
    featuresCol="tree_features",
    labelCol="delay_seconds",
    maxDepth=8,
    maxBins=64,
    seed=42
)

dt_model = dt.fit(train_tree)
dt_predictions = dt_model.transform(test_tree)

dt_mae = mae_evaluator.evaluate(dt_predictions)
dt_rmse = rmse_evaluator.evaluate(dt_predictions)
dt_r2 = r2_evaluator.evaluate(dt_predictions)

print("Decision Tree Regression")
print("MAE:", round(dt_mae, 2))
print("RMSE:", round(dt_rmse, 2))
print("R²:", round(dt_r2, 4))

Decision Tree Regression
MAE: 66.79
RMSE: 158.23
R²: 0.6756


In [9]:
from pyspark.ml.regression import RandomForestRegressor

rf = RandomForestRegressor(
    featuresCol="tree_features",
    labelCol="delay_seconds",
    numTrees=40,
    maxDepth=8,
    maxBins=64,
    seed=42
)

rf_model = rf.fit(train_tree)
rf_predictions = rf_model.transform(test_tree)

rf_mae = mae_evaluator.evaluate(rf_predictions)
rf_rmse = rmse_evaluator.evaluate(rf_predictions)
rf_r2 = r2_evaluator.evaluate(rf_predictions)

print("Random Forest Regression")
print("MAE:", round(rf_mae, 2))
print("RMSE:", round(rf_rmse, 2))
print("R²:", round(rf_r2, 4))

Random Forest Regression
MAE: 61.95
RMSE: 137.57
R²: 0.7548


In [10]:
from pyspark.ml.regression import GBTRegressor

gbt = GBTRegressor(
    featuresCol="tree_features",
    labelCol="delay_seconds",
    maxIter=40,
    maxDepth=6,
    maxBins=64,
    seed=42
)

gbt_model = gbt.fit(train_tree)
gbt_predictions = gbt_model.transform(test_tree)

gbt_mae = mae_evaluator.evaluate(gbt_predictions)
gbt_rmse = rmse_evaluator.evaluate(gbt_predictions)
gbt_r2 = r2_evaluator.evaluate(gbt_predictions)

print("Gradient-Boosted Tree Regression")
print("MAE:", round(gbt_mae, 2))
print("RMSE:", round(gbt_rmse, 2))
print("R²:", round(gbt_r2, 4))

Gradient-Boosted Tree Regression
MAE: 66.55
RMSE: 152.51
R²: 0.6987


In [11]:
results = [
    ("Linear Regression", lr_mae, lr_rmse, lr_r2),
    ("Decision Tree", dt_mae, dt_rmse, dt_r2),
    ("Random Forest", rf_mae, rf_rmse, rf_r2),
    ("Gradient-Boosted Tree", gbt_mae, gbt_rmse, gbt_r2)
]

results_df = spark.createDataFrame(
    results,
    ["Model", "MAE", "RMSE", "R2"]
)

results_df.orderBy("RMSE").show(truncate=False)

+---------------------+-----------------+------------------+------------------+
|Model                |MAE              |RMSE              |R2                |
+---------------------+-----------------+------------------+------------------+
|Random Forest        |61.95432597038554|137.57265063208413|0.7548080614344291|
|Linear Regression    |93.33748225318786|149.35644667718233|0.7110052701325705|
|Gradient-Boosted Tree|66.54593455578396|152.50912386610483|0.6986760654103044|
|Decision Tree        |66.7926834229045 |158.23116564204238|0.6756409407117547|
+---------------------+-----------------+------------------+------------------+



In [12]:
MODEL_PATH = r"D:\Big Data Programming Project\Final Assignment\models\random_forest_delay_model"

rf_model.write() \
    .overwrite() \
    .save(MODEL_PATH)

print("Best model: Random Forest")
print("MAE:", round(rf_mae, 2))
print("RMSE:", round(rf_rmse, 2))
print("R²:", round(rf_r2, 4))
print("Model saved to:", MODEL_PATH)

Best model: Random Forest
MAE: 61.95
RMSE: 137.57
R²: 0.7548
Model saved to: D:\Big Data Programming Project\Final Assignment\models\random_forest_delay_model
